# From two h5ad files to a transport report

The whole workflow on files: a reference `.h5ad` with a donor column, a state column, raw
counts and an embedding, and a query `.h5ad` with the same genes, its own donors and the query
embedded in the reference's latent space (for example by scArches surgery or Symphony). Here the
two files are written from the simulator first.

In [1]:
import tempfile
import warnings
from pathlib import Path

from statetrace.datasets import simulate_transport

workdir = Path(tempfile.mkdtemp(prefix="statetrace_tutorial_"))
warnings.formatwarning = (
    lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"
)
ref_sim, query_sim, _ = simulate_transport(
    n_ref_groups=8,
    n_query_groups=6,
    n_states=5,
    n_latent=10,
    n_genes=300,
    shifted_states={"S1": 9.0},
    absent_states=["S4"],
    minority_states={"S3": 0.25},
    seed=5,
)
ref_sim.uns.clear()
query_sim.uns.clear()
ref_sim.write_h5ad(workdir / "reference.h5ad")
query_sim.write_h5ad(workdir / "query.h5ad")

## The workflow

Eleven calls, no project-specific code (the usability target is at most 15):

In [2]:
import anndata as ad

import statetrace as st

reference = ad.read_h5ad(workdir / "reference.h5ad")
query = ad.read_h5ad(workdir / "query.h5ad")

ref = st.Reference(
    reference, group_key="donor_id", state_key="cell_state", latent_key="X_latent"
).fit()
ref.leave_one_group_out()
frozen = ref.freeze(workdir / "reference.statetrace")

route = st.validate_transport(st.Reference.load(frozen).fit(), query, group_key="donor_id")
report = route.report(workdir / "report")
print(route.summary())

StateTrace transport (knn) over 'donor_id': 1200 query cells, 6 groups, 5 reference states
  status: supported=2, supported_with_caution=0, shifted=1, insufficient=1, absent=1
  cells: supported 0.755 | uncertain 0.000 | out_of_reference 0.245


In [3]:
sorted(p.name for p in report.iterdir())

['cells.csv.gz', 'config.json', 'groups.csv', 'report.md', 'states.csv']

In [4]:
print((report / "report.md").read_text())

# StateTrace transport report

Backend: **knn** | query groups (`donor_id`): 6 | query cells: 1200 | reference states: 5 | created 2026-10-15T14:23:37+00:00 | StateTrace 0.1.0

## Transport status

**supported** 2, **supported_with_caution** 0, **shifted** 1, **insufficient** 1, **absent** 1

States marked `supported` have enough group-level evidence to enter downstream differential-abundance, pseudobulk or genetic analyses; `supported_with_caution` states should be reported with their caveat; `shifted`, `insufficient` and `absent` states should not be analysed as the reference construct.

## Evidence per state

| state | ref_groups | ref_reproducibility | groups_recovered | recovery_rate | median_confidence | uncertain_fraction | ood_fraction | shift_raw/residual | shift_level | shift_source | groups_shifted | novel_population | status | caution_reasons |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| S0 | 8 | stable | 6/6 | 1.00 | 1.00 | 0.00 | 0.00 | 6.5/-0.4 | low 